In [5]:
import torch
from rdkit import Chem
from torch_geometric.data import Data, Batch

def molecule_to_graph(smiles, label=0.0):
    """Zet een SMILES string om in een PyTorch Geometric Data object."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # 1. Knooppunten (Nodes): We gebruiken het atoomnummer als wiskundig kenmerk
    node_features = []
    for atom in mol.GetAtoms():
        node_features.append([atom.GetAtomicNum()])
    x = torch.tensor(node_features, dtype=torch.float)
    
    # 2. Verbindingen (Edges): Welke atomen zitten aan elkaar vast?
    edges = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        # Een binding werkt twee kanten op in een ongerichte graaf
        edges.append([i, j])
        edges.append([j, i])
        
    if len(edges) > 0:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        
    # 3. Doelwaarde (Label)
    y = torch.tensor([[label]], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, y=y)

print("✅ Functie 'molecule_to_graph' en bibliotheken succesvol geladen!")

✅ Functie 'molecule_to_graph' en bibliotheken succesvol geladen!


In [6]:
from torch_geometric.data import Batch

# 1. Het molecuul dat we willen testen: Ethanol
smiles_test = "CCO"
naam = "Ethanol"

# 2. Zet de SMILES code om naar een graaf (getallen-matrix)
# We geven '0.0' als label mee, want het echte label weten we zogenaamd niet
graph_test = molecule_to_graph(smiles_test, label=0.0)

if graph_test is None:
    print("Fout: Kon geen geldige graaf maken van dit molecuul.")
else:
    # 3. PyTorch modellen verwachten altijd een "batch" (een groepje data). 
    # We maken hier een groepje van 1 molecuul.
    batch_test = Batch.from_data_list([graph_test]).to(device)
    
    # 4. Zet het model in evaluatie-modus en voorspel!
    model.eval()
    with torch.no_grad():
        voorspelling_geschaald = model(batch_test)
        
    # 5. Het model spuugt een geschaald getal uit. Dit moeten we terugrekenen 
    # naar de échte scheikundige eenheid met onze 'scaler'
    voorspelling_echt = scaler.inverse_transform(voorspelling_geschaald.cpu().numpy())[0][0]
    
    print(f"🧪 Molecuul: {naam}")
    print(f"🧬 SMILES:   {smiles_test}")
    print(f"💧 Voorspelde Oplosbaarheid (LogS): {voorspelling_echt:.3f}")
    
    # Een klein hulpmiddel voor de interpretatie:
    if voorspelling_echt > -1.0:
        print("➡️ Conclusie: Lost uitstekend op in water! (Hydrofiel)")
    elif voorspelling_echt > -3.0:
        print("➡️ Conclusie: Lost matig op in water.")
    else:
        print("➡️ Conclusie: Lost zeer slecht op in water. (Lipofiel/Vetachtig)")
    # Sla de gewichten (het geleerde brein) op naar je harde schijf
    torch.save(model.state_dict(), "mijn_oplosbaarheid_model.pth")

NameError: name 'device' is not defined